# CLARA S04: captures to surrogate to optimiser

Three programs that share only the feature store and the model registry:

1. **Feature**: Cheetah captures of the S04 section into `clara_s04_captures`.
2. **Training**: feature view `clara_s04_surrogate`, MLP surrogate, registered with the view.
3. **Inference**: Bayesian optimisation against the deployed surrogate, one Cheetah check written back as `bo_check`.

Real: the CLARA lattice, the archiver magnet ranges, the reference tune, the current conversion (all from `challenges/optimisation`). Synthetic: every capture, flagged `synthetic=True`.

`LOCAL = True` runs everything on the laptop with parquet files. Set it to `False` on the cluster.

In [ ]:
LOCAL = True
from pathlib import Path
DATA, MODEL_DIR = Path("data"), Path("model")

import pandas as pd
import clara, feature_pipeline, training_pipeline, inference_pipeline

## 1. Feature pipeline

Eight knobs, as the real SETI PVs: four quadrupoles and two H/V corrector pairs. Each is drawn inside the range it actually ran over six months of operation.

In [ ]:
ref = clara.reference_currents()
pd.DataFrame([{"knob": k, "low_A": lo, "high_A": hi, "reference_A": ref[k]} for k, (lo, hi) in clara.bounds().items()]).round(3)

In [ ]:
df = feature_pipeline.campaign(runs=2000, name="s04_a", seed=1, beam_seed=100, spread=1.0, noise_fraction=0.0)
df[clara.FEATURES[:2] + clara.LABELS + ["beam_lost", "lattice_hash", "cheetah_version"]].head()

In [ ]:
if LOCAL:
    DATA.mkdir(exist_ok=True)
    df.to_parquet(DATA / "s04_a.parquet", index=False)
else:
    feature_pipeline.insert(df)

## 2. Training pipeline

The feature view keeps captures where the beam reached the screen. Scaling sits inside the model, so serving needs only the pickle.

In [ ]:
if LOCAL:
    X_train, X_test, y_train, y_test = training_pipeline.local_split(DATA)
else:
    project, fv, (X_train, X_test, y_train, y_test) = training_pipeline.feature_store_split(version=1)

model, metrics = training_pipeline.fit(X_train[clara.FEATURES], X_test[clara.FEATURES], y_train[clara.LABELS], y_test[clara.LABELS])
pd.Series(metrics).round(3)

In [ ]:
import joblib
MODEL_DIR.mkdir(exist_ok=True)
joblib.dump(model, MODEL_DIR / "model.pkl")
if not LOCAL:
    project.get_model_registry().python.create_model(
        name=training_pipeline.MODEL, metrics=metrics, feature_view=fv, input_example=X_train[clara.FEATURES].head(1),
    ).save(str(MODEL_DIR))

## 3. Inference pipeline

The optimiser spends its 80 evaluations on the surrogate. The machine (here Cheetah) sees one setting: the best one.

In [ ]:
if LOCAL:
    predict = lambda settings: model.predict(pd.DataFrame(settings)[clara.FEATURES])
else:
    predict = inference_pipeline.deployed_predict(model_version=1)

best, predicted = inference_pipeline.optimise(predict, n_calls=80, spread=1.0, seed=0)
check = inference_pipeline.check(best, beam_seed=100)
print(f"surrogate objective {predicted:.1f} um, cheetah objective {clara.objective_um(check[clara.LABELS].iloc[0]):.1f} um")
check[clara.LABELS]

In [ ]:
# Close the loop: the checked run becomes training data for the next surrogate.
if LOCAL:
    check.to_parquet(DATA / "bo_check.parquet", index=False)
else:
    feature_pipeline.insert(check)